In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.feature_extraction.text import TfidfVectorizer
from mealpy.swarm_based import PSO
from mealpy.utils.problem import FloatVar
import warnings
warnings.filterwarnings('ignore')

# ----------------------------
# Load your Sentiment dataset
# ----------------------------
def load_data():
    amazon = pd.read_csv("amazon_cells_labelled.txt", sep="\t", header=None, names=["text", "label"])
    imdb   = pd.read_csv("imdb_labelled.txt", sep="\t", header=None, names=["text", "label"])
    yelp   = pd.read_csv("yelp_labelled.txt", sep="\t", header=None, names=["text", "label"])

    df = pd.concat([amazon, imdb, yelp], axis=0)

    vectorizer = TfidfVectorizer(stop_words="english", max_features=1000)
    X = vectorizer.fit_transform(df["text"]).toarray()
    y = df["label"].values

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    return X_train, X_test, y_train, y_test

# ----------------------------
# Objective function for PSO
# ----------------------------
def objective_function(solution):
    global X_train, y_train

    n_estimators = int(solution[0])
    max_depth = int(solution[1]) if solution[1] > 0 else None
    min_samples_split = int(solution[2])
    min_samples_leaf = int(solution[3])
    max_features = solution[4]

    try:
        rf = RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            max_features=max_features,
            random_state=42,
            n_jobs=-1
        )
        scores = cross_val_score(rf, X_train, y_train, cv=3, scoring='accuracy')
        fitness = -np.mean(scores)  # Negative because optimizer minimizes
    except Exception:
        fitness = 1.0  # Penalty if invalid

    return fitness

# ----------------------------
# Run PSO optimization
# ----------------------------
def optimize_random_forest():
    global X_train, X_test, y_train, y_test
    X_train, X_test, y_train, y_test = load_data()

    print("Starting Random Forest optimization with Mealpy PSO...")
    print(f"Training set size: {X_train.shape}")
    print(f"Test set size: {X_test.shape}")
    print("-" * 50)

    # Problem bounds for hyperparameters
    problem = {
        "bounds": [
            FloatVar(lb=10, ub=200, name="n_estimators"),
            FloatVar(lb=1, ub=20, name="max_depth"),
            FloatVar(lb=2, ub=20, name="min_samples_split"),
            FloatVar(lb=1, ub=10, name="min_samples_leaf"),
            FloatVar(lb=0.1, ub=1.0, name="max_features")
        ],
        "minmax": "min",
        "obj_func": objective_function
    }

    optimizer = PSO.OriginalPSO(epoch=5, pop_size=10)
    best_agent = optimizer.solve(problem)  # Returns Agent object
    best_position = best_agent.solution          # Correct attribute for params
    best_fitness = best_agent.target.fitness     # Extract numeric float

    print("\nOptimization Results:")
    print("-" * 50)
    print(f"Best fitness (negative accuracy): {best_fitness:.6f}")
    print(f"Best accuracy: {-best_fitness:.6f}")

    best_params = {
        'n_estimators': int(best_position[0]),
        'max_depth': int(best_position[1]) if best_position[1] > 0 else None,
        'min_samples_split': int(best_position[2]),
        'min_samples_leaf': int(best_position[3]),
        'max_features': best_position[4],
        'random_state': 42
    }

    print("\nBest Hyperparameters:")
    for param, value in best_params.items():
        print(f"  {param}: {value}")

    return best_params

# ----------------------------
# Evaluate optimized model + KNN
# ----------------------------
def evaluate_models(best_params):
    global X_train, X_test, y_train, y_test

    print("\n" + "="*50)
    print("FINAL MODEL EVALUATION")
    print("="*50)

    # Optimized Random Forest
    best_rf = RandomForestClassifier(**best_params)
    best_rf.fit(X_train, y_train)
    y_pred = best_rf.predict(X_test)
    rf_acc = accuracy_score(y_test, y_pred)

    print(f"\nOptimized Random Forest Accuracy: {rf_acc:.6f}")
    print("Classification Report (Random Forest):")
    print(classification_report(y_test, y_pred, target_names=['Negative', 'Positive']))

    # Baseline KNN
    knn = KNeighborsClassifier(n_neighbors=5)
    knn.fit(X_train, y_train)
    knn_pred = knn.predict(X_test)
    knn_acc = accuracy_score(y_test, knn_pred)

    print(f"\nBaseline KNN Accuracy: {knn_acc:.6f}")
    print("Classification Report (KNN):")
    print(classification_report(y_test, knn_pred, target_names=['Negative', 'Positive']))

    # Comparison
    print("\n" + "="*50)
    print("MODEL COMPARISON")
    print("="*50)
    print(f"KNN Accuracy: {knn_acc:.6f}")
    print(f"Optimized Random Forest Accuracy: {rf_acc:.6f}")
    print(f"Difference: {rf_acc - knn_acc:.6f}")

# ----------------------------
# Main Execution
# ----------------------------
if __name__ == "__main__":
    best_params = optimize_random_forest()
    evaluate_models(best_params)
    print("\n" + "="*50)
    print("OPTIMIZATION COMPLETE!")
    print("="*50)


,text,label
0,So there is no way for me to plug it in here i...,0
1,"Good case, Excellent value.",1
2,Great for the jawbone.,1
3,Tied to charger for conversations lasting more...,0
4,The mic is great.,1
